![hslu_logo.png](img/hslu_logo.png)

## Week 5

<hr style="border:1px solid black">


# Excercise: Object Detection using YOLO v8

---
---

This exercise illustrates the inference of a YOLO v8 [1] detection model.

[1] Terven, J., Córdova-Esparza, D. M., & Romero-González, J. A. (2023). A Comprehensive Review of YOLO Architectures in Computer Vision: From YOLOv1 to YOLOv8 and YOLO-NAS. Machine Learning and Knowledge Extraction, 5(4), 1680-1716.0-1716.

#### YOLO V8 Architecture
<figure>
  <img src="./img/yolov8_architecture.png" style="width:1000px">
  <figcaption>Source: https://github.com/ultralytics/ultralytics/issues/189</figcaption>
</figure>

### Import Packages

In [ ]:
import torch
import torchvision
import numpy as np
import matplotlib.pyplot as plt
import cv2
import time
#yolo specific
from ultralytics import YOLO
from ultralytics.data.augment import LetterBox
from ultralytics.utils import ops

from utils import plot_img

### Load Example Image

In [ ]:
img_name = './sample_img/tennis.jpg'
img = cv2.imread(img_name)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
print(img.shape)
plot_img(img, figure_size=(6,6))

### Load / Download Model

In [ ]:
model = YOLO('yolov8n.pt')

### Preprocessing

Preprocessing serves to prepare input images for 'feeding' the neural network. The main processes involved in the YOLO v8 preprocessing are:

1. Resizing and Padding: Adjusting image dimensions to be divisible by 32, ensuring compatibility with the model's architecture that includes multiple pooling layers.
2. Format Transformation: Adding a batch dimension and rearranging the data from BHWC (Batch, Height, Width, Channels) to BCHW (Batch, Channels, Height, Width) format to align with the network's input structure.
3. Data Conversion and Normalization: Converting images to PyTorch tensors, changing the data type to 32-bit floating point, and normalizing pixel values between 0 and 1. This standardizes the input for optimal processing by the neural network.

In [ ]:
letterbox = LetterBox(new_shape=320, auto=True)
img_letter = letterbox(image=img) # resizing and padding image

plot_img(img_letter, figure_size=(6,6))

In [ ]:
img_trans = img_letter[np.newaxis, ...].transpose((0, 3, 1, 2))  #BHWC to BCHW, (n, 3, h, w)
img_trans = np.ascontiguousarray(img_trans)  # contiguous (to one block of memory)
img_trans = torch.from_numpy(img_trans) # numpy array to tensor
img_trans = img_trans.float()  # uint8 to fp32
img_trans /= 255  # 0 - 255 to 0.0 - 1.0

print(f'input shape: {img_trans.shape}')

### Forward Pass

In the YOLO v8 model, a type of convolutional neural network (CNN), the forward pass involves processing the input image to predict object locations and classifications. The output for each grid cell in the feature map includes:

1. Bounding Box Location: Four values specifying the position and size of each detected object.
2. Class Confidence Scores: Eighty values indicating the probability of each object belonging to one of the model's predefined classes.

This process efficiently analyzes the entire image, utilizing the model's parameters to detect and classify objects in a structured manner.ner.f an image.

In [ ]:
preds, output_raw = model.model._predict_once(img_trans)

print(f'prediction shape: {preds.shape}')

### Post-processing

Post-processing in the YOLO v8 model is critical for refining the raw predictions into usable results. The main steps include:

1. Non-Maximum Suppression (NMS): This technique filters out overlapping bounding boxes, ensuring that each detected object is represented by a single, most accurate bounding box. It involves selecting the boxes with the highest confidence scores (above a threshold, e.g., 0.25) and suppressing nearby boxes based on a threshold for the Intersection over Union (IoU, set at 0.7 here). The process is limited to a maximum number of detections (e.g., 300) and can be class-specific or class-agnostic.
2. Scaling Bounding Boxes: The coordinates of the bounding boxes are adjusted to match the original dimensions of the input image. This scaling corrects any distortions or changes in aspect ratio introduced during the initial resizing and padding in preprocessing.

The output format after the post-processing is:
```[[x0, y0, x1, y1, confidence, category index], ...]```

In [ ]:
preds = ops.non_max_suppression(prediction=preds,
                                conf_thres=0.25,
                                iou_thres=0.7,
                                agnostic=False,
                                max_det=300,
                                classes=None)[0]

preds[:, :4] = ops.scale_boxes(img1_shape=img_trans.shape[2:], 
                               boxes=preds[:, :4], 
                               img0_shape=img.shape)

print(f'prediction shape: {preds.shape}')
print(f'\nprediction:\n{preds}')

### Detection Visualization

In [ ]:
def draw_detections(preds, image_array, class_names):

    for pred in preds:
        box = pred[:4]
        box = box.tolist()
        conf = pred[4]
        label_index = int(pred[5])
        label = class_names[label_index]
        line_width = max(round(sum(image_array.shape) / 2 * 0.003), 2) 
        font_scale=line_width / 3
        font_thickness=max(line_width - 1, 1)
        color = (255,56,56)
        txt_color = (255,255,255)
        label = f'{label}: {100*conf:.0f}%'
        p1, p2 = (int(box[0]), int(box[1])), (int(box[2]), int(box[3]))
        
        cv2.rectangle(img=image_array, 
                      pt1=p1, 
                      pt2=p2, 
                      color=color, 
                      thickness=line_width, 
                      lineType=cv2.LINE_AA)
        w, h = cv2.getTextSize(text=label, 
                               fontFace=cv2.FONT_HERSHEY_SIMPLEX,
                               fontScale=font_scale, 
                               thickness=1)[0]
        p2 = p1[0] + w, p1[1] - h - 2 
        cv2.rectangle(img=image_array, 
                      pt1=p1, 
                      pt2=p2, 
                      color=color, 
                      thickness=-1, 
                      lineType=cv2.LINE_AA)
        cv2.putText(img=image_array,
                    text=label,
                    org=(p1[0], p1[1] - 2),
                    fontFace=cv2.FONT_HERSHEY_SIMPLEX,
                    fontScale=font_scale,
                    color=txt_color,
                    thickness=font_thickness,
                    lineType=cv2.LINE_AA)
    
    return image_array

In [ ]:
image_vis = draw_detections(preds, img.copy(), model.names)

plot_img(image_vis, figure_size=(6,6))